In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

In [ ]:
# Load ctcf table
# df = pd.read_csv(
#     "/scratch1/smaruj/suppressing_CTCFs/results_repeated/preexisting_ctcf_df.tsv", 
#     sep="\t"
# )

# df = pd.read_csv(
#     "/scratch1/smaruj/suppressing_CTCFs/results_repeated/new_ctcf_df.tsv", 
#     sep="\t"
# )

df = pd.read_csv(
    "/home1/smaruj/akitaV2-analyses/input_data/preprocess_boundary_CTCFs/output/CTCFs_jaspar_filtered_mm10.tsv", 
    sep="\t"
)

In [ ]:
df

In [ ]:
def reverse_complement_tensor(seq_tensor):
    """
    Reverse complement a sequence tensor.
    seq_tensor shape: [1, 4, L] or [4, L]
    Assumes [A, C, G, T] encoding
    """
    if seq_tensor.dim() == 3:
        seq_tensor = seq_tensor.squeeze(0)  # [4, L]
    
    # Reverse along sequence dimension
    seq_rc = torch.flip(seq_tensor, dims=[1])
    # Complement: [A, C, G, T] -> [T, G, C, A]
    seq_rc = seq_rc[[3, 2, 1, 0], :]
    
    return seq_rc.unsqueeze(0)  # [1, 4, L]

In [ ]:
def tensor_to_sequence(seq_tensor):
    """Convert one-hot tensor to DNA sequence string."""
    if seq_tensor.dim() == 3:
        seq_tensor = seq_tensor.squeeze(0)  # [4, L]
    
    nucleotides = ['A', 'C', 'G', 'T']
    indices = torch.argmax(seq_tensor, dim=0)
    sequence = ''.join([nucleotides[i] for i in indices])
    return sequence

def calculate_gc_content(sequence):
    """Calculate GC content from a DNA sequence string."""
    sequence = sequence.upper()
    gc_count = sequence.count('G') + sequence.count('C')
    total = len(sequence)
    return gc_count / total if total > 0 else 0.0

In [ ]:
device = torch.device("cpu")
bin_size = 2048
center_bin = 320

slice_start = bin_size * center_bin
slice_end = bin_size * (center_bin + 1)

In [ ]:
# Add columns for GC content
df['gc_core'] = None
df['gc_upstream_15'] = None
df['gc_upstream_30'] = None
df['gc_downstream_15'] = None
df['gc_downstream_30'] = None
df['gc_core_plus_15'] = None
df['gc_core_plus_30'] = None

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Calculating GC content"):
    fold = row['fold']
    chrom = row['chrom']
    centered_start = row['centered_start']
    centered_end = row['centered_end']
    ctcf_start = row['ctcf_start']
    ctcf_end = row['ctcf_end']
    orientation = row['orientation']
    
    # Load slice
    slice_path = f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/fold{fold}/{chrom}_{centered_start}_{centered_end}_slice.pt"
    if not os.path.exists(slice_path):
        continue
    
    slice_ohe = torch.load(slice_path, map_location=device)
    
    # Load full sequence
    full_path = f"/scratch1/smaruj/suppressing_CTCFs/ohe_X/fold{fold}/{chrom}_{centered_start}_{centered_end}_X.pt"
    if not os.path.exists(full_path):
        continue
    
    full_ohe = torch.load(full_path, map_location=device)
    
    # Extract with 30bp flanks
    extract_start = ctcf_start - 30
    extract_end = ctcf_end + 30
    
    # Extract sequence
    if extract_start >= 0 and extract_end <= 2048:
        # Fully inside slice
        ctcf_seq = slice_ohe[:, :, extract_start:extract_end]
    else:
        parts = []
        
        # Left overhang
        if extract_start < 0:
            left_len = -extract_start
            parts.append(full_ohe[:, :, slice_start + extract_start : slice_start])
            parts.append(slice_ohe[:, :, 0:extract_end])
        # Right overhang
        elif extract_end > 2048:
            right_len = extract_end - 2048
            parts.append(slice_ohe[:, :, extract_start:2048])
            parts.append(full_ohe[:, :, slice_end : slice_end + right_len])
        else:
            ctcf_seq = slice_ohe[:, :, extract_start:extract_end]
            parts = []
        
        if parts:
            ctcf_seq = torch.cat(parts, dim=2)
    
    # Reverse complement if needed
    if orientation == '-':
        ctcf_seq = reverse_complement_tensor(ctcf_seq)
    
    # Convert to sequence string
    full_seq = tensor_to_sequence(ctcf_seq)
    
    # Extract regions (core is in the middle with 30bp flanks on each side)
    core_length = ctcf_end - ctcf_start
    core_start_idx = 30
    core_end_idx = 30 + core_length
    
    df.at[idx, 'gc_core'] = calculate_gc_content(full_seq[core_start_idx:core_end_idx])
    df.at[idx, 'gc_upstream_15'] = calculate_gc_content(full_seq[core_start_idx-15:core_start_idx])
    df.at[idx, 'gc_upstream_30'] = calculate_gc_content(full_seq[core_start_idx-30:core_start_idx])
    df.at[idx, 'gc_downstream_15'] = calculate_gc_content(full_seq[core_end_idx:core_end_idx+15])
    df.at[idx, 'gc_downstream_30'] = calculate_gc_content(full_seq[core_end_idx:core_end_idx+30])
    df.at[idx, 'gc_core_plus_15'] = calculate_gc_content(full_seq[core_start_idx-15:core_end_idx+15])
    df.at[idx, 'gc_core_plus_30'] = calculate_gc_content(full_seq[core_start_idx-30:core_end_idx+30])

print("\nGC content calculated!")
print(df[['gc_core', 'gc_core_plus_15', 'gc_core_plus_30', 
          'gc_upstream_15', 'gc_downstream_15']].describe())

In [ ]:
# output_path = f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/preexisting_ctcf_df_with_gc.tsv"
output_path = f"/scratch1/smaruj/suppressing_CTCFs/results_repeated/new_ctcf_df_with_gc.tsv"
df.to_csv(output_path, sep="\t", index=False)